## Introduction: Delivering Business Value with GenAI

Delivering business value with GenAI requires answering these questions:
1. **Is my agent producing accurate answers?**
2. **How do I improve my agent's accuracy?**
3. **Is my agent fast and cost effective?**

We know how to deliver reliable software, but GenAI's non-deterministic nature makes it difficult to directly apply software development best practices:
1. **User inputs evolve without warning**
2. **Human expertise required to assess output quality**
3. **Must trade-off between quality & cost/latency**

**MLflow enables you to apply software development best practices to evaluate and monitor GenAI application quality.**

![MLflow SLDC](https://i.imgur.com/T0uM1No.gif)

### What you will see

- **Implement MLflow Tracing** to observe and debug your GenAI app
- **Customize MLflow's LLM judges** to create quality evaluation criteria that align with business requirements and your domain expert's judgement
- **Create MLflow Evaluation Datasets** to curate production traces into test suites
- **Use MLflow's Evaluation SDK** to iteratively test changes and check for regressions
- **Use MLflow Prompt Registry and App Versioning** to link versions to quality evaluations
- **Use Databricks AI/BI** to link GenAI observability and evaluation metrics to business KPIs

![MLflow GenAI Demo](https://i.imgur.com/MXhaayF.gif)

## Install packages (only required if running in a Databricks Notebook)

In [ ]:
%pip install -U -r ../requirements.txt
dbutils.library.restartPython()

## Environment Setup

Let's start by setting up the environment and checking that everything is working correctly.

In [ ]:
import sys
sys.path.append('../')
sys.path.append('../../')

import os
import mlflow
from mlflow_demo.utils import *

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

# Configure for the current participant: derives the object prefix, prompt name,
# model service, and experiment from YOUR username. The widgets only carry the
# shared catalog/schema/warehouse.
setup_databricks_notebook_env()

print('=== Environment Setup (derived from your username) ===')
print(f"prefix:        {os.environ['UC_TOOL_PREFIX']}")
print(f"catalog:       {os.environ['UC_CATALOG']}.{os.environ['UC_SCHEMA']}")
print(f"prompt:        {os.environ['PROMPT_NAME']}")
print(f"model service: {os.environ['LLM_MODEL_SERVICE']}")
print(f"experiment:    {os.environ['MLFLOW_EXPERIMENT_NAME']} (ID {os.environ['MLFLOW_EXPERIMENT_ID']})")
print(f"warehouse:     {os.environ['MLFLOW_TRACING_SQL_WAREHOUSE_ID']}")

import logging
logging.getLogger("urllib3").setLevel(logging.ERROR)
logging.getLogger("mlflow").setLevel(logging.ERROR)

## Use Case Overview: Banking Customer Support Agent

We'll create a **Banking Customer Support Agent** -- a tool-calling agent that helps bank customers with account, transaction, loan, credit card, and product questions via Unity Catalog functions. The goal is to drive business KPIs such as customer satisfaction, first-contact resolution rate, and cost per interaction.

This use case demonstrates real-world challenges:
- **Accuracy with real financial data** - The agent must query UC functions for actual account balances, transactions, and loan details
- **Compliance and safety** - Responses must not expose sensitive data or provide unauthorized financial advice
- **Business impact measurement** - Link agent quality to customer satisfaction and resolution rates

## Interactive Demo Navigation

Follow these steps to learn MLflow GenAI evaluation best practices through hands-on examples.

### Step Navigation

Click the buttons below to navigate to each step of the demo:

In [ ]:
# Interactive navigation cards linking to each demo notebook.
steps_info = [
    {
        "title": "Step 1: Observe with Tracing",
        "description": "Add observability to your GenAI app so you can see what's happening and collect user feedback",
        "notebook": "1_observe_with_traces.ipynb",
    },
    {
        "title": "Step 2: Create Quality Metrics",
        "description": "Scale your expert's judgment using MLflow's LLM judges to create automated quality metrics",
        "notebook": "2_create_quality_metrics.ipynb",
    },
    {
        "title": "Step 3: Find & Fix Quality Issues",
        "description": "Use production traces and actual user data with evaluation metrics to iteratively test quality fixes",
        "notebook": "3_find_fix_quality_issues.ipynb",
    },
    {
        "title": "Step 4: Human Review",
        "description": "Collect expert feedback through an intuitive UI to systematically improve GenAI quality",
        "notebook": "4_human_review.ipynb",
    },
    {
        "title": "Step 5: Production Monitoring",
        "description": "Monitor GenAI quality in production with intelligent sampling",
        "notebook": "5_production_monitoring.ipynb",
    },
]


def _notebook_url(notebook_name):
    """Best-effort link to a sibling notebook. Builds the Databricks workspace
    URL from the current notebook's path when running in a notebook; falls back
    to a relative link when running locally."""
    stem = notebook_name[:-6] if notebook_name.endswith(".ipynb") else notebook_name
    try:
        import posixpath

        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        host = ctx.browserHostName().get()
        current = ctx.notebookPath().get()
        sibling = posixpath.join(posixpath.dirname(current), stem)
        return f"https://{host}/#workspace{sibling}"
    except Exception:
        return f"./{notebook_name}"


def create_step_card(step_num, title, description, notebook_name):
    notebook_url = _notebook_url(notebook_name)
    card_html = f"""
    <div style="border: 1px solid #e0e0e0; border-radius: 8px; padding: 20px; margin: 10px 0; background: #f9f9f9;">
        <div style="display: flex; align-items: center; margin-bottom: 10px;">
            <div style="width: 40px; height: 40px; border-radius: 50%; background: #007acc; color: white; display: flex; align-items: center; justify-content: center; font-weight: bold; margin-right: 15px;">{step_num}</div>
            <h3 style="margin: 0; color: #333;">{title}</h3>
        </div>
        <p style="color: #666; margin: 10px 0;">{description}</p>
        <a href="{notebook_url}" target="_blank" style="background: #007acc; color: white; padding: 8px 16px; border-radius: 4px; text-decoration: none;">Open {notebook_name}</a>
    </div>
    """
    return card_html

if mlflow.utils.databricks_utils.is_in_databricks_notebook():
    all_cards_html = "<h3>Interactive Demo Steps</h3>"
    for i, step in enumerate(steps_info, 1):
        all_cards_html += create_step_card(i, step["title"], step["description"], step["notebook"])
    displayHTML(all_cards_html)
else:
    step_cards = []
    for i, step in enumerate(steps_info, 1):
        card_html = create_step_card(i, step["title"], step["description"], step["notebook"])
        step_cards.append(widgets.HTML(card_html))
    display(widgets.VBox([widgets.HTML("<h3>Interactive Demo Steps</h3>"), *step_cards]))

---

## Additional Resources

### Documentation
- [MLflow GenAI Documentation](https://docs.databricks.com/aws/en/mlflow3/genai/)
- [MLflow Tracing Guide](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/)
- [MLflow Evaluation SDK](https://docs.databricks.com/aws/en/mlflow3/genai/evaluation/)

### What You'll Learn
By completing this demo, you'll understand how to:
- Implement comprehensive GenAI observability with MLflow Tracing
- Create custom quality metrics using LLM judges
- Build automated evaluation pipelines for continuous quality monitoring
- Collect and utilize expert feedback for quality improvement
- Link technical metrics to business KPIs and ROI

### Need Help?
If you encounter any issues:
1. Check that your environment variables are properly set
2. Ensure you have the necessary permissions in your Databricks workspace
3. Restart the Python kernel if needed
4. Refer to the individual notebook instructions for step-specific guidance

---

**Happy learning!**

## Exercise

Write 3 banking questions you'd want to test against this agent.

Consider edge cases such as:
- Questions about specific transaction disputes
- Questions that require combining data from multiple tools (accounts + transactions + loans)
- Questions where the customer's data might not exist